In [ ]:
# ============================================================
# PHASE 8E.4–8E.7
# BMCS benchmarks, XGBoost selection, probability calibration,
# shadow threshold selection, and untouched test evaluation.
#
# Run in a Snowflake Python Worksheet.
#
# Handler: main
# Return type: String
#
# Required packages:
#   pandas
#   scikit-learn
#   xgboost
#
# Synthetic data only. All outputs remain SHADOW.
# ============================================================

from __future__ import annotations

import json
from datetime import datetime, timezone
from typing import Any, Callable

import numpy as np
import pandas as pd

from snowflake.snowpark import Session

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

from xgboost import XGBClassifier


DATABASE = "KMAT_COST_MODEL_DB"
SCHEMA = "CORE_ML"
SOURCE_OBJECT = (
    "KMAT_COST_MODEL_DB.CORE_ML.BMCS_MODEL_DATASET_V2"
)

ENTITY_COL = "ENTITY_ID"
RFQ_COL = "RFQ_ID"
TIMESTAMP_COL = "FEATURE_AS_OF_TIMESTAMP"
SPLIT_COL = "DATA_SPLIT"
TARGET_COL = "TARGET_CORRECT_MAPPING_FLAG"

RULE_SCORE_COL = "RULE_BMCS_SCORE"
CORTEX_SCORE_COL = "CORTEX_CONFIDENCE_SCORE"
HYBRID_SCORE_COL = "DEVELOPMENT_HYBRID_BMCS_SCORE"

RANDOM_STATE = 42
PROBABILITY_EPSILON = 1e-6

AUTO_APPROVAL_MAX_ERROR_RATE = 0.02
AUTO_REJECTION_MAX_ERROR_RATE = 0.05
MIN_POLICY_ROWS = 50

MODEL_FIT_END = pd.Timestamp("2024-07-01")
MODEL_SELECTION_END = pd.Timestamp("2025-01-01")
CALIBRATION_END = pd.Timestamp("2025-04-01")
THRESHOLD_SELECTION_END = pd.Timestamp("2025-07-01")


CATEGORICAL_COLS = [
    "SOURCE_LANGUAGE",
    "EXTRACTED_ENGINE",
    "EXTRACTED_CAB",
    "EXTRACTED_WHEEL",
    "EXTRACTED_COLOR",
    "CORTEX_EXTRACTION_STATUS",
]

NUMERIC_COLS = [
    "DOCUMENT_LENGTH_CHARS",
    "PAGE_COUNT",
    "OCR_QUALITY_SCORE",
    "DOCUMENT_COMPLEXITY_SCORE",
    "AMBIGUITY_TERM_COUNT",
    "CONFLICTING_PHRASE_COUNT",
    "UNSUPPORTED_VALUE_COUNT",
    "SYNONYM_MATCH_STRENGTH",
    "TRANSLATION_USED_FLAG",
    "TRANSLATION_QUALITY_SCORE",
    "SOURCE_TEXT_PRESENT_FLAG",
    "ENGINE_EXTRACTION_CONFIDENCE",
    "CAB_EXTRACTION_CONFIDENCE",
    "WHEEL_EXTRACTION_CONFIDENCE",
    "COLOR_EXTRACTION_CONFIDENCE",
    "ENGINE_TEXT_EVIDENCE_FLAG",
    "CAB_TEXT_EVIDENCE_FLAG",
    "WHEEL_TEXT_EVIDENCE_FLAG",
    "COLOR_TEXT_EVIDENCE_FLAG",
    "VALID_EXTRACTED_VALUE_COUNT",
    "TEXT_SUPPORTED_VALUE_COUNT",
    "MISSING_REQUIREMENT_COUNT",
    "EXACT_PHRASE_MATCH_COUNT",
    "SYNONYM_MATCH_COUNT",
    "RULE_VALIDATION_PASS_COUNT",
    "CORTEX_REASONING_AVAILABLE_FLAG",
    RULE_SCORE_COL,
    CORTEX_SCORE_COL,
    HYBRID_SCORE_COL,
]

MODEL_FEATURE_COLS = CATEGORICAL_COLS + NUMERIC_COLS


def utc_now_text() -> str:
    return datetime.now(timezone.utc).strftime(
        "%Y-%m-%d %H:%M:%S.%f"
    )


def clip_probability(values: np.ndarray) -> np.ndarray:
    return np.clip(
        np.asarray(values, dtype=float),
        PROBABILITY_EPSILON,
        1.0 - PROBABILITY_EPSILON,
    )


def expected_calibration_error(
    actual: np.ndarray,
    probability: np.ndarray,
    bin_count: int = 10,
) -> float:
    actual = np.asarray(actual, dtype=int)
    probability = clip_probability(probability)

    edges = np.linspace(0.0, 1.0, bin_count + 1)
    bin_ids = np.digitize(
        probability,
        edges[1:-1],
        right=True,
    )

    total = len(actual)
    ece = 0.0

    for bin_id in range(bin_count):
        mask = bin_ids == bin_id

        if not mask.any():
            continue

        observed_rate = float(actual[mask].mean())
        mean_probability = float(probability[mask].mean())

        ece += (
            float(mask.sum()) / total
        ) * abs(observed_rate - mean_probability)

    return float(ece)


def probability_metrics(
    actual: np.ndarray,
    probability: np.ndarray,
    threshold: float = 0.50,
) -> dict[str, float | int]:
    actual = np.asarray(actual, dtype=int)
    probability = clip_probability(probability)
    predicted = (probability >= threshold).astype(int)

    return {
        "ROW_COUNT": int(len(actual)),
        "CORRECT_MAPPING_RATE": float(actual.mean()),
        "ROC_AUC": float(roc_auc_score(actual, probability)),
        "PR_AUC": float(
            average_precision_score(actual, probability)
        ),
        "BRIER_SCORE": float(
            brier_score_loss(actual, probability)
        ),
        "LOG_LOSS": float(
            log_loss(actual, probability, labels=[0, 1])
        ),
        "ECE_10_BIN": expected_calibration_error(
            actual,
            probability,
            bin_count=10,
        ),
        "ACCURACY_AT_050": float(
            accuracy_score(actual, predicted)
        ),
        "PRECISION_AT_050": float(
            precision_score(
                actual,
                predicted,
                zero_division=0,
            )
        ),
        "RECALL_AT_050": float(
            recall_score(
                actual,
                predicted,
                zero_division=0,
            )
        ),
        "F1_AT_050": float(
            f1_score(
                actual,
                predicted,
                zero_division=0,
            )
        ),
        "PROBABILITY_MIN": float(probability.min()),
        "PROBABILITY_MAX": float(probability.max()),
        "NULL_PROBABILITY_ROWS": int(
            np.isnan(probability).sum()
        ),
    }


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def make_preprocessor(
    scale_numeric: bool,
) -> ColumnTransformer:
    numeric_transformer: Any = (
        StandardScaler()
        if scale_numeric
        else "passthrough"
    )

    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                make_one_hot_encoder(),
                CATEGORICAL_COLS,
            ),
            (
                "numeric",
                numeric_transformer,
                NUMERIC_COLS,
            ),
        ],
        remainder="drop",
    )


def build_candidates() -> dict[str, Pipeline]:
    return {
        "LOGISTIC_REGRESSION": Pipeline(
            steps=[
                (
                    "preprocess",
                    make_preprocessor(scale_numeric=True),
                ),
                (
                    "model",
                    LogisticRegression(
                        max_iter=1500,
                        C=1.0,
                        class_weight=None,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),

        "RANDOM_FOREST_V1": Pipeline(
            steps=[
                (
                    "preprocess",
                    make_preprocessor(scale_numeric=False),
                ),
                (
                    "model",
                    RandomForestClassifier(
                        n_estimators=300,
                        max_depth=15,
                        min_samples_leaf=5,
                        max_features=0.75,
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                    ),
                ),
            ]
        ),

        "XGB_V1_BALANCED": Pipeline(
            steps=[
                (
                    "preprocess",
                    make_preprocessor(scale_numeric=False),
                ),
                (
                    "model",
                    XGBClassifier(
                        objective="binary:logistic",
                        eval_metric="logloss",
                        n_estimators=350,
                        learning_rate=0.035,
                        max_depth=5,
                        min_child_weight=5,
                        subsample=0.85,
                        colsample_bytree=0.85,
                        reg_alpha=0.05,
                        reg_lambda=1.25,
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                        tree_method="hist",
                    ),
                ),
            ]
        ),

        "XGB_V2_CONSERVATIVE": Pipeline(
            steps=[
                (
                    "preprocess",
                    make_preprocessor(scale_numeric=False),
                ),
                (
                    "model",
                    XGBClassifier(
                        objective="binary:logistic",
                        eval_metric="logloss",
                        n_estimators=450,
                        learning_rate=0.025,
                        max_depth=4,
                        min_child_weight=8,
                        subsample=0.90,
                        colsample_bytree=0.90,
                        reg_alpha=0.10,
                        reg_lambda=1.75,
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                        tree_method="hist",
                    ),
                ),
            ]
        ),

        "XGB_V3_REGULARIZED": Pipeline(
            steps=[
                (
                    "preprocess",
                    make_preprocessor(scale_numeric=False),
                ),
                (
                    "model",
                    XGBClassifier(
                        objective="binary:logistic",
                        eval_metric="logloss",
                        n_estimators=400,
                        learning_rate=0.03,
                        max_depth=6,
                        min_child_weight=7,
                        subsample=0.82,
                        colsample_bytree=0.82,
                        reg_alpha=0.20,
                        reg_lambda=2.0,
                        random_state=RANDOM_STATE,
                        n_jobs=1,
                        tree_method="hist",
                    ),
                ),
            ]
        ),
    }


def validate_source(source: pd.DataFrame) -> None:
    required = {
        ENTITY_COL,
        RFQ_COL,
        TIMESTAMP_COL,
        SPLIT_COL,
        TARGET_COL,
        RULE_SCORE_COL,
        CORTEX_SCORE_COL,
        HYBRID_SCORE_COL,
        *MODEL_FEATURE_COLS,
    }

    missing = sorted(required.difference(source.columns))

    if missing:
        raise ValueError(
            f"Missing BMCS model columns: {missing}"
        )

    if source.empty:
        raise ValueError("BMCS model dataset is empty.")

    if source[ENTITY_COL].isna().any():
        raise ValueError("ENTITY_ID contains null values.")

    if source[ENTITY_COL].duplicated().any():
        raise ValueError(
            "ENTITY_ID contains duplicate records."
        )

    if source[TARGET_COL].isna().any():
        raise ValueError("BMCS target contains null values.")

    invalid_labels = ~source[TARGET_COL].isin([0, 1])

    if invalid_labels.any():
        raise ValueError(
            "BMCS target contains values other than 0 or 1."
        )

    feature_nulls = source[
        MODEL_FEATURE_COLS
    ].isna().sum()
    feature_nulls = feature_nulls[
        feature_nulls > 0
    ]

    if not feature_nulls.empty:
        raise ValueError(
            "BMCS features contain null values: "
            + str(feature_nulls.to_dict())
        )

    actual_splits = set(
        source[SPLIT_COL].dropna().unique()
    )
    expected_splits = {
        "TRAIN",
        "VALIDATION",
        "TEST",
    }

    if actual_splits != expected_splits:
        raise ValueError(
            "Expected TRAIN, VALIDATION and TEST; "
            f"found {actual_splits}."
        )


def baseline_probability(
    source: pd.DataFrame,
    score_col: str,
) -> np.ndarray:
    return clip_probability(
        source[score_col].astype(float).to_numpy()
        / 100.0
    )


def model_selection_score(
    metrics: pd.DataFrame,
) -> pd.Series:
    return (
        0.40 * (1.0 - metrics["PR_AUC"])
        + 0.25 * metrics["BRIER_SCORE"]
        + 0.20 * (1.0 - metrics["ROC_AUC"])
        + 0.10 * metrics["LOG_LOSS"]
        + 0.05 * metrics["ECE_10_BIN"]
    )


def fit_calibrators(
    raw_probability: np.ndarray,
    actual: np.ndarray,
) -> dict[str, Callable[[np.ndarray], np.ndarray]]:
    raw_probability = clip_probability(raw_probability)
    actual = np.asarray(actual, dtype=int)

    raw_logit = np.log(
        raw_probability / (1.0 - raw_probability)
    ).reshape(-1, 1)

    platt = LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=RANDOM_STATE,
    )
    platt.fit(raw_logit, actual)

    isotonic = IsotonicRegression(
        y_min=PROBABILITY_EPSILON,
        y_max=1.0 - PROBABILITY_EPSILON,
        out_of_bounds="clip",
    )
    isotonic.fit(raw_probability, actual)

    def identity(values: np.ndarray) -> np.ndarray:
        return clip_probability(values)

    def platt_predict(values: np.ndarray) -> np.ndarray:
        values = clip_probability(values)
        logits = np.log(
            values / (1.0 - values)
        ).reshape(-1, 1)
        return clip_probability(
            platt.predict_proba(logits)[:, 1]
        )

    def isotonic_predict(
        values: np.ndarray,
    ) -> np.ndarray:
        return clip_probability(
            isotonic.predict(
                clip_probability(values)
            )
        )

    return {
        "NONE": identity,
        "PLATT": platt_predict,
        "ISOTONIC": isotonic_predict,
    }


def build_threshold_candidates(
    actual: np.ndarray,
    probability: np.ndarray,
) -> pd.DataFrame:
    actual = np.asarray(actual, dtype=int)
    probability = clip_probability(probability)
    rows: list[dict[str, Any]] = []
    total = len(actual)

    high_thresholds = np.round(
        np.arange(0.50, 1.000, 0.005),
        3,
    )

    for threshold in high_thresholds:
        approved = probability >= threshold
        approved_count = int(approved.sum())
        incorrect_count = int(
            ((actual == 0) & approved).sum()
        )

        rows.append(
            {
                "POLICY_TYPE": "AUTO_APPROVE",
                "THRESHOLD": float(threshold),
                "POLICY_ROW_COUNT": approved_count,
                "POLICY_COVERAGE_RATE": (
                    approved_count / total
                ),
                "ERROR_ROW_COUNT": incorrect_count,
                "ERROR_RATE_WITHIN_POLICY": (
                    incorrect_count / approved_count
                    if approved_count > 0
                    else None
                ),
                "ERROR_RATE_IN_POPULATION": (
                    incorrect_count / total
                ),
                "POLICY_ELIGIBLE_FLAG": (
                    approved_count >= MIN_POLICY_ROWS
                    and (
                        incorrect_count / approved_count
                        if approved_count > 0
                        else 1.0
                    )
                    <= AUTO_APPROVAL_MAX_ERROR_RATE
                ),
            }
        )

    low_thresholds = np.round(
        np.arange(0.005, 0.505, 0.005),
        3,
    )

    for threshold in low_thresholds:
        rejected = probability <= threshold
        rejected_count = int(rejected.sum())
        incorrect_rejection_count = int(
            ((actual == 1) & rejected).sum()
        )

        rows.append(
            {
                "POLICY_TYPE": "AUTO_REJECT",
                "THRESHOLD": float(threshold),
                "POLICY_ROW_COUNT": rejected_count,
                "POLICY_COVERAGE_RATE": (
                    rejected_count / total
                ),
                "ERROR_ROW_COUNT":
                    incorrect_rejection_count,
                "ERROR_RATE_WITHIN_POLICY": (
                    incorrect_rejection_count
                    / rejected_count
                    if rejected_count > 0
                    else None
                ),
                "ERROR_RATE_IN_POPULATION": (
                    incorrect_rejection_count / total
                ),
                "POLICY_ELIGIBLE_FLAG": (
                    rejected_count >= MIN_POLICY_ROWS
                    and (
                        incorrect_rejection_count
                        / rejected_count
                        if rejected_count > 0
                        else 1.0
                    )
                    <= AUTO_REJECTION_MAX_ERROR_RATE
                ),
            }
        )

    return pd.DataFrame(rows)


def select_threshold_policy(
    candidates: pd.DataFrame,
) -> dict[str, Any]:
    auto_approve = candidates[
        (candidates["POLICY_TYPE"] == "AUTO_APPROVE")
        & candidates["POLICY_ELIGIBLE_FLAG"]
    ].copy()

    if auto_approve.empty:
        high_threshold = 1.0
        high_status = (
            "NO_ELIGIBLE_AUTO_APPROVAL_THRESHOLD"
        )
    else:
        auto_approve = auto_approve.sort_values(
            by=[
                "POLICY_COVERAGE_RATE",
                "ERROR_RATE_WITHIN_POLICY",
                "THRESHOLD",
            ],
            ascending=[False, True, False],
        )
        high_threshold = float(
            auto_approve.iloc[0]["THRESHOLD"]
        )
        high_status = "AUTO_APPROVAL_THRESHOLD_SELECTED"

    auto_reject = candidates[
        (candidates["POLICY_TYPE"] == "AUTO_REJECT")
        & candidates["POLICY_ELIGIBLE_FLAG"]
    ].copy()

    if auto_reject.empty:
        low_threshold = 0.0
        low_status = (
            "NO_ELIGIBLE_AUTO_REJECTION_THRESHOLD"
        )
    else:
        auto_reject = auto_reject.sort_values(
            by=[
                "POLICY_COVERAGE_RATE",
                "ERROR_RATE_WITHIN_POLICY",
                "THRESHOLD",
            ],
            ascending=[False, True, False],
        )
        low_threshold = float(
            auto_reject.iloc[0]["THRESHOLD"]
        )
        low_status = "AUTO_REJECTION_THRESHOLD_SELECTED"

    if low_threshold >= high_threshold:
        low_threshold = 0.0
        low_status = (
            "AUTO_REJECTION_DISABLED_THRESHOLD_OVERLAP"
        )

    return {
        "LOW_THRESHOLD": low_threshold,
        "HIGH_THRESHOLD": high_threshold,
        "LOW_THRESHOLD_STATUS": low_status,
        "HIGH_THRESHOLD_STATUS": high_status,
    }


def trust_action(
    probability: np.ndarray,
    low_threshold: float,
    high_threshold: float,
) -> np.ndarray:
    probability = clip_probability(probability)

    return np.select(
        [
            probability >= high_threshold,
            probability <= low_threshold,
        ],
        [
            "AUTO_APPROVE_CANDIDATE",
            "AUTO_REJECT_CANDIDATE",
        ],
        default="ENGINEERING_REVIEW_REQUIRED",
    )


def policy_metrics(
    actual: np.ndarray,
    probability: np.ndarray,
    low_threshold: float,
    high_threshold: float,
) -> dict[str, float | int]:
    actual = np.asarray(actual, dtype=int)
    probability = clip_probability(probability)

    approved = probability >= high_threshold
    rejected = probability <= low_threshold
    reviewed = ~(approved | rejected)

    incorrect_approved = (
        approved & (actual == 0)
    )
    correct_rejected = (
        rejected & (actual == 1)
    )

    approved_count = int(approved.sum())
    rejected_count = int(rejected.sum())

    return {
        "AUTO_APPROVAL_ROWS": approved_count,
        "AUTO_APPROVAL_RATE": float(approved.mean()),
        "INCORRECT_AUTO_APPROVAL_ROWS": int(
            incorrect_approved.sum()
        ),
        "INCORRECT_AUTO_APPROVAL_RATE": (
            float(
                incorrect_approved.sum()
                / approved_count
            )
            if approved_count > 0
            else 0.0
        ),
        "INCORRECT_AUTO_APPROVAL_POPULATION_RATE":
            float(incorrect_approved.mean()),
        "AUTO_REJECTION_ROWS": rejected_count,
        "AUTO_REJECTION_RATE": float(rejected.mean()),
        "CORRECT_MAPPING_REJECTION_ROWS": int(
            correct_rejected.sum()
        ),
        "CORRECT_MAPPING_REJECTION_RATE": (
            float(
                correct_rejected.sum()
                / rejected_count
            )
            if rejected_count > 0
            else 0.0
        ),
        "ENGINEERING_REVIEW_ROWS": int(
            reviewed.sum()
        ),
        "ENGINEERING_REVIEW_RATE": float(
            reviewed.mean()
        ),
    }


def save_table(
    session: Session,
    dataframe: pd.DataFrame,
    table_name: str,
) -> None:
    session.write_pandas(
        dataframe,
        table_name=table_name,
        database=DATABASE,
        schema=SCHEMA,
        auto_create_table=True,
        overwrite=True,
        quote_identifiers=False,
    )


def create_run_log(session: Session) -> None:
    session.sql(
        """
        CREATE TABLE IF NOT EXISTS
        KMAT_COST_MODEL_DB.CORE_ML.BMCS_ML_RUN_LOG_V2
        (
            RUN_ID VARCHAR,
            RUN_STATUS VARCHAR,
            MESSAGE VARCHAR,
            CREATED_AT TIMESTAMP_NTZ
                DEFAULT CURRENT_TIMESTAMP()
        )
        """
    ).collect()


def insert_run_log(
    session: Session,
    run_id: str,
    status: str,
    message: str,
) -> None:
    safe_message = message.replace("'", "''")[:8000]

    session.sql(
        f"""
        INSERT INTO
        KMAT_COST_MODEL_DB.CORE_ML.BMCS_ML_RUN_LOG_V2
        (
            RUN_ID,
            RUN_STATUS,
            MESSAGE
        )
        VALUES
        (
            '{run_id}',
            '{status}',
            '{safe_message}'
        )
        """
    ).collect()


def main(session: Session) -> str:
    run_id = (
        "BMCSML_"
        + datetime.now(timezone.utc).strftime(
            "%Y%m%d_%H%M%S_%f"
        )
    )

    create_run_log(session)
    insert_run_log(
        session,
        run_id,
        "STARTED",
        "BMCS candidate selection started.",
    )

    try:
        source = session.table(
            SOURCE_OBJECT
        ).to_pandas()

        source.columns = [
            str(column).upper()
            for column in source.columns
        ]

        source[TIMESTAMP_COL] = pd.to_datetime(
            source[TIMESTAMP_COL]
        )

        source[TARGET_COL] = source[
            TARGET_COL
        ].astype(int)

        validate_source(source)

        model_fit = source[
            (source[SPLIT_COL] == "TRAIN")
            & (
                source[TIMESTAMP_COL]
                < MODEL_FIT_END
            )
        ].copy()

        model_selection = source[
            (source[SPLIT_COL] == "TRAIN")
            & (
                source[TIMESTAMP_COL]
                >= MODEL_FIT_END
            )
            & (
                source[TIMESTAMP_COL]
                < MODEL_SELECTION_END
            )
        ].copy()

        calibration = source[
            (source[SPLIT_COL] == "VALIDATION")
            & (
                source[TIMESTAMP_COL]
                < CALIBRATION_END
            )
        ].copy()

        threshold_selection = source[
            (source[SPLIT_COL] == "VALIDATION")
            & (
                source[TIMESTAMP_COL]
                >= CALIBRATION_END
            )
            & (
                source[TIMESTAMP_COL]
                < THRESHOLD_SELECTION_END
            )
        ].copy()

        full_train = source[
            source[SPLIT_COL] == "TRAIN"
        ].copy()

        test = source[
            source[SPLIT_COL] == "TEST"
        ].copy()

        partition_sizes = {
            "model_fit": len(model_fit),
            "model_selection": len(model_selection),
            "calibration": len(calibration),
            "threshold_selection":
                len(threshold_selection),
            "test": len(test),
        }

        for partition_name, row_count in (
            partition_sizes.items()
        ):
            if row_count < 200:
                raise RuntimeError(
                    f"{partition_name} contains only "
                    f"{row_count} rows."
                )

        X_fit = model_fit[MODEL_FEATURE_COLS]
        y_fit = model_fit[TARGET_COL].to_numpy()

        X_select = model_selection[
            MODEL_FEATURE_COLS
        ]
        y_select = model_selection[
            TARGET_COL
        ].to_numpy()

        selection_rows: list[dict[str, Any]] = []

        rule_probability = baseline_probability(
            model_selection,
            RULE_SCORE_COL,
        )
        rule_metrics = probability_metrics(
            y_select,
            rule_probability,
        )
        selection_rows.append(
            {
                "MODEL_NAME": "RULE_BMCS_BASELINE",
                "MODEL_TYPE": "RULE_BASELINE",
                **rule_metrics,
            }
        )

        hybrid_probability = baseline_probability(
            model_selection,
            HYBRID_SCORE_COL,
        )
        hybrid_metrics = probability_metrics(
            y_select,
            hybrid_probability,
        )
        selection_rows.append(
            {
                "MODEL_NAME":
                    "RULE_CORTEX_HYBRID_BASELINE",
                "MODEL_TYPE": "HYBRID_BASELINE",
                **hybrid_metrics,
            }
        )

        candidates = build_candidates()
        fitted_candidates: dict[str, Pipeline] = {}

        for model_name, model in candidates.items():
            model.fit(X_fit, y_fit)
            probability = clip_probability(
                model.predict_proba(X_select)[:, 1]
            )

            metrics = probability_metrics(
                y_select,
                probability,
            )

            selection_rows.append(
                {
                    "MODEL_NAME": model_name,
                    "MODEL_TYPE": (
                        "XGBOOST"
                        if model_name.startswith("XGB")
                        else model_name
                    ),
                    **metrics,
                }
            )

            fitted_candidates[model_name] = model

        selection_metrics = pd.DataFrame(
            selection_rows
        )

        hybrid_row = selection_metrics[
            selection_metrics["MODEL_NAME"]
            == "RULE_CORTEX_HYBRID_BASELINE"
        ].iloc[0]

        selection_metrics[
            "HYBRID_PR_AUC_GATE_FLAG"
        ] = (
            selection_metrics["PR_AUC"]
            >= float(hybrid_row["PR_AUC"]) - 0.005
        )

        selection_metrics[
            "HYBRID_BRIER_GATE_FLAG"
        ] = (
            selection_metrics["BRIER_SCORE"]
            <= float(hybrid_row["BRIER_SCORE"])
            + 0.005
        )

        selection_metrics[
            "SELECTION_ELIGIBLE_FLAG"
        ] = (
            selection_metrics["MODEL_NAME"]
            .str.startswith("XGB")
            & selection_metrics[
                "HYBRID_PR_AUC_GATE_FLAG"
            ]
            & selection_metrics[
                "HYBRID_BRIER_GATE_FLAG"
            ]
        )

        selection_metrics["SELECTION_SCORE"] = (
            model_selection_score(
                selection_metrics
            )
        )

        eligible = selection_metrics[
            selection_metrics[
                "SELECTION_ELIGIBLE_FLAG"
            ]
        ].sort_values(
            by=[
                "SELECTION_SCORE",
                "BRIER_SCORE",
                "PR_AUC",
            ],
            ascending=[True, True, False],
        )

        if eligible.empty:
            raise RuntimeError(
                "No XGBoost candidate passed the "
                "hybrid-baseline validation gates. "
                "The test set was not evaluated."
            )

        selected_model_name = str(
            eligible.iloc[0]["MODEL_NAME"]
        )

        selection_metrics[
            "SELECTED_MODEL_FLAG"
        ] = (
            selection_metrics["MODEL_NAME"]
            == selected_model_name
        )

        selection_metrics[
            "EVALUATION_WINDOW"
        ] = "2024-07-01_TO_2024-12-31"

        selection_metrics[
            "MODEL_VERSION"
        ] = "V2"

        selection_metrics[
            "FEATURE_SET_VERSION"
        ] = "BMCS_FEATURES_V2"

        selection_metrics[
            "TRAINING_DATA_CLASS"
        ] = "SYNTHETIC"

        selection_metrics[
            "DEPLOYMENT_MODE"
        ] = "SHADOW"

        selection_metrics[
            "OFFICIAL_COST_IMPACT_ALLOWED_FLAG"
        ] = False

        selection_metrics[
            "BUSINESS_DECISION_ALLOWED_FLAG"
        ] = False

        selection_metrics[
            "EVALUATED_AT_UTC"
        ] = utc_now_text()

        save_table(
            session,
            selection_metrics,
            "BMCS_VALIDATION_MODEL_METRICS_V2",
        )

        # Retrain selected base model on the entire original TRAIN.
        final_base_model = build_candidates()[
            selected_model_name
        ]

        final_base_model.fit(
            full_train[MODEL_FEATURE_COLS],
            full_train[TARGET_COL].to_numpy(),
        )

        raw_calibration_probability = clip_probability(
            final_base_model.predict_proba(
                calibration[MODEL_FEATURE_COLS]
            )[:, 1]
        )

        calibrators = fit_calibrators(
            raw_calibration_probability,
            calibration[TARGET_COL].to_numpy(),
        )

        raw_threshold_probability = clip_probability(
            final_base_model.predict_proba(
                threshold_selection[
                    MODEL_FEATURE_COLS
                ]
            )[:, 1]
        )

        calibration_rows: list[dict[str, Any]] = []

        for calibration_method, transform in (
            calibrators.items()
        ):
            calibrated_probability = transform(
                raw_threshold_probability
            )

            metrics = probability_metrics(
                threshold_selection[
                    TARGET_COL
                ].to_numpy(),
                calibrated_probability,
            )

            calibration_rows.append(
                {
                    "MODEL_NAME":
                        selected_model_name,
                    "MODEL_VERSION": "V2",
                    "CALIBRATION_METHOD":
                        calibration_method,
                    "CALIBRATION_FIT_WINDOW":
                        "2025-01-01_TO_2025-03-31",
                    "CALIBRATION_EVALUATION_WINDOW":
                        "2025-04-01_TO_2025-06-30",
                    **metrics,
                }
            )

        calibration_metrics = pd.DataFrame(
            calibration_rows
        )

        calibration_metrics[
            "CALIBRATION_SELECTION_SCORE"
        ] = (
            0.55
            * calibration_metrics["BRIER_SCORE"]
            + 0.30
            * calibration_metrics["ECE_10_BIN"]
            + 0.15
            * calibration_metrics["LOG_LOSS"]
        )

        calibration_metrics = (
            calibration_metrics.sort_values(
                by=[
                    "CALIBRATION_SELECTION_SCORE",
                    "BRIER_SCORE",
                    "ECE_10_BIN",
                ],
                ascending=True,
            )
        )

        selected_calibration_method = str(
            calibration_metrics.iloc[0][
                "CALIBRATION_METHOD"
            ]
        )

        calibration_metrics[
            "SELECTED_CALIBRATION_FLAG"
        ] = (
            calibration_metrics[
                "CALIBRATION_METHOD"
            ]
            == selected_calibration_method
        )

        calibration_metrics[
            "TRAINING_DATA_CLASS"
        ] = "SYNTHETIC"

        calibration_metrics[
            "DEPLOYMENT_MODE"
        ] = "SHADOW"

        calibration_metrics[
            "OFFICIAL_COST_IMPACT_ALLOWED_FLAG"
        ] = False

        calibration_metrics[
            "BUSINESS_DECISION_ALLOWED_FLAG"
        ] = False

        calibration_metrics[
            "EVALUATED_AT_UTC"
        ] = utc_now_text()

        save_table(
            session,
            calibration_metrics,
            "BMCS_CALIBRATION_METRICS_V2",
        )

        selected_calibrator = calibrators[
            selected_calibration_method
        ]

        selected_threshold_probability = (
            selected_calibrator(
                raw_threshold_probability
            )
        )

        threshold_candidates = (
            build_threshold_candidates(
                threshold_selection[
                    TARGET_COL
                ].to_numpy(),
                selected_threshold_probability,
            )
        )

        threshold_candidates[
            "MODEL_NAME"
        ] = selected_model_name

        threshold_candidates[
            "MODEL_VERSION"
        ] = "V2"

        threshold_candidates[
            "CALIBRATION_METHOD"
        ] = selected_calibration_method

        threshold_candidates[
            "THRESHOLD_SELECTION_WINDOW"
        ] = "2025-04-01_TO_2025-06-30"

        threshold_candidates[
            "AUTO_APPROVAL_MAX_ERROR_RATE"
        ] = AUTO_APPROVAL_MAX_ERROR_RATE

        threshold_candidates[
            "AUTO_REJECTION_MAX_ERROR_RATE"
        ] = AUTO_REJECTION_MAX_ERROR_RATE

        threshold_candidates[
            "DEPLOYMENT_MODE"
        ] = "SHADOW"

        threshold_candidates[
            "EVALUATED_AT_UTC"
        ] = utc_now_text()

        save_table(
            session,
            threshold_candidates,
            "BMCS_THRESHOLD_CANDIDATES_V2",
        )

        selected_policy = select_threshold_policy(
            threshold_candidates
        )

        low_threshold = float(
            selected_policy["LOW_THRESHOLD"]
        )
        high_threshold = float(
            selected_policy["HIGH_THRESHOLD"]
        )

        threshold_policy_metrics = policy_metrics(
            threshold_selection[
                TARGET_COL
            ].to_numpy(),
            selected_threshold_probability,
            low_threshold,
            high_threshold,
        )

        selected_pipeline_model = (
            final_base_model.named_steps["model"]
        )

        selected_parameters = {
            key: value
            for key, value
            in selected_pipeline_model.get_params().items()
            if key
            in {
                "objective",
                "eval_metric",
                "n_estimators",
                "learning_rate",
                "max_depth",
                "min_child_weight",
                "subsample",
                "colsample_bytree",
                "reg_alpha",
                "reg_lambda",
                "random_state",
                "tree_method",
            }
        }

        selected_policy_table = pd.DataFrame(
            [
                {
                    "MODEL_NAME":
                        selected_model_name,
                    "MODEL_VERSION": "V2",
                    "FEATURE_SET_VERSION":
                        "BMCS_FEATURES_V2",
                    "MODEL_SELECTION_WINDOW":
                        "2024-07-01_TO_2024-12-31",
                    "CALIBRATION_FIT_WINDOW":
                        "2025-01-01_TO_2025-03-31",
                    "THRESHOLD_SELECTION_WINDOW":
                        "2025-04-01_TO_2025-06-30",
                    "CALIBRATION_METHOD":
                        selected_calibration_method,
                    "LOW_THRESHOLD":
                        low_threshold,
                    "HIGH_THRESHOLD":
                        high_threshold,
                    "LOW_THRESHOLD_STATUS":
                        selected_policy[
                            "LOW_THRESHOLD_STATUS"
                        ],
                    "HIGH_THRESHOLD_STATUS":
                        selected_policy[
                            "HIGH_THRESHOLD_STATUS"
                        ],
                    "AUTO_APPROVAL_MAX_ERROR_RATE":
                        AUTO_APPROVAL_MAX_ERROR_RATE,
                    "AUTO_REJECTION_MAX_ERROR_RATE":
                        AUTO_REJECTION_MAX_ERROR_RATE,
                    **threshold_policy_metrics,
                    "MODEL_PARAMETERS_JSON":
                        json.dumps(
                            selected_parameters,
                            sort_keys=True,
                        ),
                    "TRAINING_DATA_CLASS":
                        "SYNTHETIC",
                    "VALIDATION_STATUS":
                        "LOCKED_FOR_SYNTHETIC_TEST",
                    "DEPLOYMENT_MODE":
                        "SHADOW",
                    "OFFICIAL_COST_IMPACT_ALLOWED_FLAG":
                        False,
                    "BUSINESS_DECISION_ALLOWED_FLAG":
                        False,
                    "SELECTED_AT_UTC":
                        utc_now_text(),
                }
            ]
        )

        save_table(
            session,
            selected_policy_table,
            "BMCS_SELECTED_MODEL_POLICY_V2",
        )

        # ----------------------------------------------------
        # Untouched test evaluation begins only now.
        # ----------------------------------------------------
        raw_test_probability = clip_probability(
            final_base_model.predict_proba(
                test[MODEL_FEATURE_COLS]
            )[:, 1]
        )

        calibrated_test_probability = (
            selected_calibrator(
                raw_test_probability
            )
        )

        test_probability_metrics = probability_metrics(
            test[TARGET_COL].to_numpy(),
            calibrated_test_probability,
        )

        test_policy_metrics = policy_metrics(
            test[TARGET_COL].to_numpy(),
            calibrated_test_probability,
            low_threshold,
            high_threshold,
        )

        rule_test_metrics = probability_metrics(
            test[TARGET_COL].to_numpy(),
            baseline_probability(
                test,
                RULE_SCORE_COL,
            ),
        )

        hybrid_test_metrics = probability_metrics(
            test[TARGET_COL].to_numpy(),
            baseline_probability(
                test,
                HYBRID_SCORE_COL,
            ),
        )

        test_actions = trust_action(
            calibrated_test_probability,
            low_threshold,
            high_threshold,
        )

        test_predictions = test[
            [
                ENTITY_COL,
                RFQ_COL,
                TIMESTAMP_COL,
                SPLIT_COL,
                "SOURCE_LANGUAGE",
                "CORTEX_EXTRACTION_STATUS",
                "EXTRACTED_ENGINE",
                "EXTRACTED_CAB",
                "EXTRACTED_WHEEL",
                "EXTRACTED_COLOR",
                RULE_SCORE_COL,
                CORTEX_SCORE_COL,
                HYBRID_SCORE_COL,
                TARGET_COL,
            ]
        ].copy()

        test_predictions[
            "RAW_ML_CORRECT_MAPPING_PROBABILITY"
        ] = raw_test_probability

        test_predictions[
            "CALIBRATED_CORRECT_MAPPING_PROBABILITY"
        ] = calibrated_test_probability

        test_predictions[
            "ML_BMCS_SCORE"
        ] = (
            100.0
            * calibrated_test_probability
        )

        test_predictions[
            "LOW_THRESHOLD"
        ] = low_threshold

        test_predictions[
            "HIGH_THRESHOLD"
        ] = high_threshold

        test_predictions[
            "SHADOW_TRUST_ACTION"
        ] = test_actions

        test_predictions[
            "INCORRECT_AUTO_APPROVAL_FLAG"
        ] = (
            (test_actions == "AUTO_APPROVE_CANDIDATE")
            & (
                test_predictions[TARGET_COL]
                .to_numpy()
                == 0
            )
        ).astype(int)

        test_predictions[
            "CORRECT_MAPPING_REJECTION_FLAG"
        ] = (
            (test_actions == "AUTO_REJECT_CANDIDATE")
            & (
                test_predictions[TARGET_COL]
                .to_numpy()
                == 1
            )
        ).astype(int)

        test_predictions[
            "MODEL_NAME"
        ] = selected_model_name

        test_predictions[
            "MODEL_VERSION"
        ] = "V2"

        test_predictions[
            "FEATURE_SET_VERSION"
        ] = "BMCS_FEATURES_V2"

        test_predictions[
            "CALIBRATION_METHOD"
        ] = selected_calibration_method

        test_predictions[
            "OFFICIAL_TRUST_SOURCE"
        ] = "PHASE7_RULE_CORTEX_HYBRID"

        test_predictions[
            "ML_ADVISORY_ONLY_FLAG"
        ] = True

        test_predictions[
            "DEPLOYMENT_MODE"
        ] = "SHADOW"

        test_predictions[
            "OFFICIAL_COST_IMPACT_ALLOWED_FLAG"
        ] = False

        test_predictions[
            "BUSINESS_DECISION_ALLOWED_FLAG"
        ] = False

        test_predictions[
            "PREDICTED_AT_UTC"
        ] = utc_now_text()

        save_table(
            session,
            test_predictions,
            "BMCS_TEST_PREDICTIONS_V2",
        )

        test_evaluation = pd.DataFrame(
            [
                {
                    "MODEL_NAME":
                        selected_model_name,
                    "MODEL_VERSION": "V2",
                    "DATA_SPLIT": "TEST",
                    "CALIBRATION_METHOD":
                        selected_calibration_method,
                    "LOW_THRESHOLD":
                        low_threshold,
                    "HIGH_THRESHOLD":
                        high_threshold,
                    **test_probability_metrics,
                    **test_policy_metrics,
                    "RULE_BASELINE_PR_AUC":
                        rule_test_metrics["PR_AUC"],
                    "RULE_BASELINE_BRIER_SCORE":
                        rule_test_metrics[
                            "BRIER_SCORE"
                        ],
                    "HYBRID_BASELINE_PR_AUC":
                        hybrid_test_metrics["PR_AUC"],
                    "HYBRID_BASELINE_BRIER_SCORE":
                        hybrid_test_metrics[
                            "BRIER_SCORE"
                        ],
                    "PR_AUC_IMPROVEMENT_VS_HYBRID":
                        test_probability_metrics[
                            "PR_AUC"
                        ]
                        - hybrid_test_metrics[
                            "PR_AUC"
                        ],
                    "BRIER_IMPROVEMENT_VS_HYBRID":
                        hybrid_test_metrics[
                            "BRIER_SCORE"
                        ]
                        - test_probability_metrics[
                            "BRIER_SCORE"
                        ],
                    "VALIDATION_STATUS": (
                        "ACCEPTED_FOR_SHADOW"
                        if (
                            test_probability_metrics[
                                "BRIER_SCORE"
                            ]
                            < hybrid_test_metrics[
                                "BRIER_SCORE"
                            ]
                            and test_policy_metrics[
                                "INCORRECT_AUTO_APPROVAL_RATE"
                            ]
                            <= AUTO_APPROVAL_MAX_ERROR_RATE
                        )
                        else
                        "REQUIRES_MODEL_CORRECTION"
                    ),
                    "TRAINING_DATA_CLASS":
                        "SYNTHETIC",
                    "DEPLOYMENT_MODE":
                        "SHADOW",
                    "OFFICIAL_COST_IMPACT_ALLOWED_FLAG":
                        False,
                    "BUSINESS_DECISION_ALLOWED_FLAG":
                        False,
                    "EVALUATED_AT_UTC":
                        utc_now_text(),
                }
            ]
        )

        save_table(
            session,
            test_evaluation,
            "BMCS_TEST_EVALUATION_V2",
        )

        # ----------------------------------------------------
        # Language and extraction-status monitoring.
        # ----------------------------------------------------
        segment_rows: list[dict[str, Any]] = []

        for segment_type, segment_col in [
            ("SOURCE_LANGUAGE", "SOURCE_LANGUAGE"),
            (
                "CORTEX_EXTRACTION_STATUS",
                "CORTEX_EXTRACTION_STATUS",
            ),
        ]:
            for segment_value, group in (
                test_predictions.groupby(
                    segment_col,
                    dropna=False,
                )
            ):
                group_actual = group[
                    TARGET_COL
                ].to_numpy()

                group_probability = group[
                    "CALIBRATED_CORRECT_MAPPING_PROBABILITY"
                ].to_numpy()

                probability_result = probability_metrics(
                    group_actual,
                    group_probability,
                )

                policy_result = policy_metrics(
                    group_actual,
                    group_probability,
                    low_threshold,
                    high_threshold,
                )

                segment_rows.append(
                    {
                        "MODEL_NAME":
                            selected_model_name,
                        "MODEL_VERSION": "V2",
                        "SEGMENT_TYPE":
                            segment_type,
                        "SEGMENT_VALUE":
                            str(segment_value),
                        **probability_result,
                        **policy_result,
                        "DEPLOYMENT_MODE":
                            "SHADOW",
                        "EVALUATED_AT_UTC":
                            utc_now_text(),
                    }
                )

        segment_metrics = pd.DataFrame(
            segment_rows
        )

        save_table(
            session,
            segment_metrics,
            "BMCS_TEST_SEGMENT_METRICS_V2",
        )

        # ----------------------------------------------------
        # XGBoost feature importance.
        # ----------------------------------------------------
        preprocessor = (
            final_base_model.named_steps[
                "preprocess"
            ]
        )
        xgb_model = (
            final_base_model.named_steps[
                "model"
            ]
        )

        transformed_feature_names = (
            preprocessor.get_feature_names_out()
        )

        feature_importance = pd.DataFrame(
            {
                "FEATURE_NAME":
                    transformed_feature_names,
                "IMPORTANCE_SCORE":
                    xgb_model.feature_importances_,
            }
        ).sort_values(
            by="IMPORTANCE_SCORE",
            ascending=False,
        )

        feature_importance[
            "IMPORTANCE_RANK"
        ] = np.arange(
            1,
            len(feature_importance) + 1,
        )

        feature_importance[
            "MODEL_NAME"
        ] = selected_model_name

        feature_importance[
            "MODEL_VERSION"
        ] = "V2"

        feature_importance[
            "FEATURE_SET_VERSION"
        ] = "BMCS_FEATURES_V2"

        feature_importance[
            "CALCULATED_AT_UTC"
        ] = utc_now_text()

        save_table(
            session,
            feature_importance,
            "BMCS_FEATURE_IMPORTANCE_V2",
        )

        success_message = (
            f"selected_model={selected_model_name}; "
            f"calibration={selected_calibration_method}; "
            f"low_threshold={low_threshold}; "
            f"high_threshold={high_threshold}; "
            f"test_rows={len(test)}"
        )

        insert_run_log(
            session,
            run_id,
            "SUCCESS",
            success_message,
        )

        return json.dumps(
            {
                "status": "SUCCESS",
                "selected_model":
                    selected_model_name,
                "selected_calibration":
                    selected_calibration_method,
                "low_threshold":
                    low_threshold,
                "high_threshold":
                    high_threshold,
                "partition_sizes":
                    partition_sizes,
                "test_evaluation_table": (
                    "KMAT_COST_MODEL_DB.CORE_ML."
                    "BMCS_TEST_EVALUATION_V2"
                ),
                "test_predictions_table": (
                    "KMAT_COST_MODEL_DB.CORE_ML."
                    "BMCS_TEST_PREDICTIONS_V2"
                ),
                "deployment_mode": "SHADOW",
                "official_cost_impact_allowed":
                    False,
                "business_decision_allowed":
                    False,
            },
            sort_keys=True,
        )

    except Exception as exc:
        insert_run_log(
            session,
            run_id,
            "FAILED",
            f"{type(exc).__name__}: {str(exc)}",
        )
        raise

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE KMAT_COST_MODEL_DB").collect()
session.sql("USE SCHEMA CORE_ML").collect()

result = main(session)
print(result)

In [ ]:
#Next step is to register the calibrated BMCS classifier as a physical Snowflake model 
#Create the Phase 7 RFQ shadow-scoring integration
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import sklearn, xgboost
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.isotonic import IsotonicRegression
from xgboost import XGBClassifier
from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

DB="KMAT_COST_MODEL_DB"
SCHEMA="CORE_ML"
MODEL_NAME="BMCS_CALIBRATED_CLASSIFIER"
VERSION_NAME="V2_SHADOW_1"
SOURCE=f"{DB}.{SCHEMA}.BMCS_MODEL_DATASET_V2"

CATS=["SOURCE_LANGUAGE","EXTRACTED_ENGINE","EXTRACTED_CAB","EXTRACTED_WHEEL","EXTRACTED_COLOR","CORTEX_EXTRACTION_STATUS"]
NUMS=["DOCUMENT_LENGTH_CHARS","PAGE_COUNT","OCR_QUALITY_SCORE","DOCUMENT_COMPLEXITY_SCORE","AMBIGUITY_TERM_COUNT","CONFLICTING_PHRASE_COUNT","UNSUPPORTED_VALUE_COUNT","SYNONYM_MATCH_STRENGTH","TRANSLATION_USED_FLAG","TRANSLATION_QUALITY_SCORE","SOURCE_TEXT_PRESENT_FLAG","ENGINE_EXTRACTION_CONFIDENCE","CAB_EXTRACTION_CONFIDENCE","WHEEL_EXTRACTION_CONFIDENCE","COLOR_EXTRACTION_CONFIDENCE","ENGINE_TEXT_EVIDENCE_FLAG","CAB_TEXT_EVIDENCE_FLAG","WHEEL_TEXT_EVIDENCE_FLAG","COLOR_TEXT_EVIDENCE_FLAG","VALID_EXTRACTED_VALUE_COUNT","TEXT_SUPPORTED_VALUE_COUNT","MISSING_REQUIREMENT_COUNT","EXACT_PHRASE_MATCH_COUNT","SYNONYM_MATCH_COUNT","RULE_VALIDATION_PASS_COUNT","CORTEX_REASONING_AVAILABLE_FLAG","RULE_BMCS_SCORE","CORTEX_CONFIDENCE_SCORE","DEVELOPMENT_HYBRID_BMCS_SCORE"]
FEATURES=CATS+NUMS

def encoder():
    try: return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError: return OneHotEncoder(handle_unknown="ignore", sparse=False)

def build_pipeline():
    prep=ColumnTransformer([("categorical",encoder(),CATS),("numeric","passthrough",NUMS)],remainder="drop")
    clf=XGBClassifier(objective="binary:logistic",eval_metric="logloss",n_estimators=350,learning_rate=0.035,max_depth=5,min_child_weight=5,subsample=0.85,colsample_bytree=0.85,reg_alpha=0.05,reg_lambda=1.25,random_state=42,n_jobs=1,tree_method="hist")
    return Pipeline([("preprocess",prep),("model",clf)])

def main(session: Session) -> str:
    session.use_database(DB); session.use_schema(SCHEMA)
    pdf=session.table(SOURCE).to_pandas()
    pdf.columns=[c.upper() for c in pdf.columns]
    pdf["FEATURE_AS_OF_TIMESTAMP"]=pd.to_datetime(pdf["FEATURE_AS_OF_TIMESTAMP"])
    train=pdf[pdf["DATA_SPLIT"]=="TRAIN"].copy()
    calibration=pdf[(pdf["DATA_SPLIT"]=="VALIDATION") & (pdf["FEATURE_AS_OF_TIMESTAMP"]<pd.Timestamp("2025-04-01"))].copy()

    base=build_pipeline()
    base.fit(train[FEATURES],train["TARGET_CORRECT_MAPPING_FLAG"].astype(int))

    # Fit isotonic calibrator and save parameters for post-inference application
    raw_cal=np.clip(base.predict_proba(calibration[FEATURES])[:,1],1e-6,1-1e-6)
    iso=IsotonicRegression(y_min=1e-6,y_max=1-1e-6,out_of_bounds="clip")
    iso.fit(raw_cal,calibration["TARGET_CORRECT_MAPPING_FLAG"].astype(int))

    # Save calibrator parameters as a table for post-inference calibration
    cal_params=pd.DataFrame({"X_THRESHOLDS":iso.X_thresholds_,"Y_THRESHOLDS":iso.y_thresholds_})
    session.write_pandas(cal_params, table_name="BMCS_ISOTONIC_CALIBRATOR_PARAMS_V2",
                         database=DB, schema=SCHEMA, auto_create_table=True, overwrite=True, quote_identifiers=False)

    # Log the base pipeline directly (predict_proba supported natively)
    reg=Registry(session=session,database_name=DB,schema_name=SCHEMA)

    # Drop existing model to re-register cleanly
    session.sql(f"DROP MODEL IF EXISTS {DB}.{SCHEMA}.{MODEL_NAME}").collect()

    metrics=session.table(f"{DB}.{SCHEMA}.BMCS_TEST_EVALUATION_V2").to_pandas().iloc[0].to_dict()
    mv=reg.log_model(
        base,
        model_name=MODEL_NAME,
        version_name=VERSION_NAME,
        comment="BMCS XGB_V1_BALANCED base classifier. Isotonic calibration applied post-inference. Synthetic shadow advisory only.",
        metrics={k:(float(v) if isinstance(v,(np.floating,float)) else v) for k,v in metrics.items() if k in ["ROC_AUC","PR_AUC","BRIER_SCORE","LOG_LOSS","ECE_10_BIN"]},
        sample_input_data=train[FEATURES].head(100),
        target_platforms=["WAREHOUSE"],
        options={"target_methods":["predict_proba"]}
    )
    action="REGISTERED"

    smoke=mv.run(train[FEATURES].head(5),function_name="predict_proba")
    return json.dumps({"status":"SUCCESS","action":action,"model":MODEL_NAME,"version":VERSION_NAME,
                       "smoke_rows":len(smoke),"calibration_method":"ISOTONIC_POST_INFERENCE",
                       "deployment_mode":"SHADOW","business_decision_allowed":False},sort_keys=True)

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
result = main(session)
print(result)

In [ ]:
# ============================================================
# PHASE 8E.8 FINAL
# Register one complete BMCS trust model containing:
#   1. One-hot preprocessing
#   2. XGB_V1_BALANCED classifier
#   3. Isotonic probability calibration
#
# Existing version V2_SHADOW_1 remains as audit history.
# This worksheet creates V2_SHADOW_2 and makes it DEFAULT
# only after a successful warehouse inference smoke test.
#
# Snowflake Python Worksheet settings:
#   Handler: main
#   Return type: String
#
# Required packages:
#   pandas
#   numpy
#   scikit-learn
#   xgboost
#   snowflake-ml-python
# ============================================================

import json
from datetime import datetime, timezone
from typing import Any, Optional

import numpy as np
import pandas as pd
import sklearn
import xgboost

from sklearn.compose import ColumnTransformer
from sklearn.isotonic import IsotonicRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from snowflake.snowpark import Session
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry


DATABASE = "KMAT_COST_MODEL_DB"
SCHEMA = "CORE_ML"

SOURCE_OBJECT = (
    "KMAT_COST_MODEL_DB.CORE_ML.BMCS_MODEL_DATASET_V2"
)

MODEL_NAME = "BMCS_CALIBRATED_CLASSIFIER"
NEW_VERSION_NAME = "V2_SHADOW_2"

TIMESTAMP_COLUMN = "FEATURE_AS_OF_TIMESTAMP"
TARGET_COLUMN = "TARGET_CORRECT_MAPPING_FLAG"

MODEL_TRAIN_END = pd.Timestamp("2025-01-01")
CALIBRATION_START = pd.Timestamp("2025-01-01")
CALIBRATION_END = pd.Timestamp("2025-04-01")

PROBABILITY_EPSILON = 1e-6

CATEGORICAL_COLUMNS = [
    "SOURCE_LANGUAGE",
    "EXTRACTED_ENGINE",
    "EXTRACTED_CAB",
    "EXTRACTED_WHEEL",
    "EXTRACTED_COLOR",
    "CORTEX_EXTRACTION_STATUS",
]

NUMERIC_COLUMNS = [
    "DOCUMENT_LENGTH_CHARS",
    "PAGE_COUNT",
    "OCR_QUALITY_SCORE",
    "DOCUMENT_COMPLEXITY_SCORE",
    "AMBIGUITY_TERM_COUNT",
    "CONFLICTING_PHRASE_COUNT",
    "UNSUPPORTED_VALUE_COUNT",
    "SYNONYM_MATCH_STRENGTH",
    "TRANSLATION_USED_FLAG",
    "TRANSLATION_QUALITY_SCORE",
    "SOURCE_TEXT_PRESENT_FLAG",
    "ENGINE_EXTRACTION_CONFIDENCE",
    "CAB_EXTRACTION_CONFIDENCE",
    "WHEEL_EXTRACTION_CONFIDENCE",
    "COLOR_EXTRACTION_CONFIDENCE",
    "ENGINE_TEXT_EVIDENCE_FLAG",
    "CAB_TEXT_EVIDENCE_FLAG",
    "WHEEL_TEXT_EVIDENCE_FLAG",
    "COLOR_TEXT_EVIDENCE_FLAG",
    "VALID_EXTRACTED_VALUE_COUNT",
    "TEXT_SUPPORTED_VALUE_COUNT",
    "MISSING_REQUIREMENT_COUNT",
    "EXACT_PHRASE_MATCH_COUNT",
    "SYNONYM_MATCH_COUNT",
    "RULE_VALIDATION_PASS_COUNT",
    "CORTEX_REASONING_AVAILABLE_FLAG",
    "RULE_BMCS_SCORE",
    "CORTEX_CONFIDENCE_SCORE",
    "DEVELOPMENT_HYBRID_BMCS_SCORE",
]

FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


def build_base_pipeline() -> Pipeline:
    preprocessing = ColumnTransformer(
        transformers=[
            (
                "categorical",
                make_one_hot_encoder(),
                CATEGORICAL_COLUMNS,
            ),
            (
                "numeric",
                "passthrough",
                NUMERIC_COLUMNS,
            ),
        ],
        remainder="drop",
    )

    classifier = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=350,
        learning_rate=0.035,
        max_depth=5,
        min_child_weight=5,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.05,
        reg_lambda=1.25,
        random_state=42,
        n_jobs=1,
        tree_method="hist",
    )

    return Pipeline(
        steps=[
            ("preprocess", preprocessing),
            ("classifier", classifier),
        ]
    )


class GovernedCalibratedBMCSModel(
    custom_model.CustomModel
):
    """
    One atomic inference artifact.

    The registered predict method returns:
      - raw XGBoost probability
      - isotonic calibrated probability
      - BMCS score on the 0-100 display scale

    It does not assign an official business decision.
    """

    FEATURE_COLUMNS = [
        "SOURCE_LANGUAGE",
        "EXTRACTED_ENGINE",
        "EXTRACTED_CAB",
        "EXTRACTED_WHEEL",
        "EXTRACTED_COLOR",
        "CORTEX_EXTRACTION_STATUS",
        "DOCUMENT_LENGTH_CHARS",
        "PAGE_COUNT",
        "OCR_QUALITY_SCORE",
        "DOCUMENT_COMPLEXITY_SCORE",
        "AMBIGUITY_TERM_COUNT",
        "CONFLICTING_PHRASE_COUNT",
        "UNSUPPORTED_VALUE_COUNT",
        "SYNONYM_MATCH_STRENGTH",
        "TRANSLATION_USED_FLAG",
        "TRANSLATION_QUALITY_SCORE",
        "SOURCE_TEXT_PRESENT_FLAG",
        "ENGINE_EXTRACTION_CONFIDENCE",
        "CAB_EXTRACTION_CONFIDENCE",
        "WHEEL_EXTRACTION_CONFIDENCE",
        "COLOR_EXTRACTION_CONFIDENCE",
        "ENGINE_TEXT_EVIDENCE_FLAG",
        "CAB_TEXT_EVIDENCE_FLAG",
        "WHEEL_TEXT_EVIDENCE_FLAG",
        "COLOR_TEXT_EVIDENCE_FLAG",
        "VALID_EXTRACTED_VALUE_COUNT",
        "TEXT_SUPPORTED_VALUE_COUNT",
        "MISSING_REQUIREMENT_COUNT",
        "EXACT_PHRASE_MATCH_COUNT",
        "SYNONYM_MATCH_COUNT",
        "RULE_VALIDATION_PASS_COUNT",
        "CORTEX_REASONING_AVAILABLE_FLAG",
        "RULE_BMCS_SCORE",
        "CORTEX_CONFIDENCE_SCORE",
        "DEVELOPMENT_HYBRID_BMCS_SCORE",
    ]

    def __init__(
        self,
        context: custom_model.ModelContext,
    ) -> None:
        super().__init__(context)

    @custom_model.inference_api
    def predict(
        self,
        input_df: pd.DataFrame,
    ) -> pd.DataFrame:
        missing_columns = sorted(
            set(self.FEATURE_COLUMNS)
            .difference(input_df.columns)
        )

        if missing_columns:
            raise ValueError(
                "BMCS inference is missing required columns: "
                f"{missing_columns}"
            )

        features = input_df[
            self.FEATURE_COLUMNS
        ].copy()

        raw_probability = np.asarray(
            self.context[
                "base_pipeline"
            ].predict_proba(features)[:, 1],
            dtype=float,
        )

        raw_probability = np.clip(
            raw_probability,
            1e-6,
            1.0 - 1e-6,
        )

        calibrated_probability = np.asarray(
            self.context[
                "isotonic_calibrator"
            ].predict(raw_probability),
            dtype=float,
        )

        calibrated_probability = np.clip(
            calibrated_probability,
            1e-6,
            1.0 - 1e-6,
        )

        return pd.DataFrame(
            {
                "RAW_CORRECT_MAPPING_PROBABILITY":
                    raw_probability,

                "CALIBRATED_CORRECT_MAPPING_PROBABILITY":
                    calibrated_probability,

                "ML_BMCS_SCORE":
                    calibrated_probability * 100.0,
            }
        )


def create_registration_log(
    session: Session,
) -> None:
    session.sql(
        """
        CREATE TABLE IF NOT EXISTS
        KMAT_COST_MODEL_DB.CORE_ML.
        BMCS_COMPOSITE_REGISTRATION_LOG_V2
        (
            RUN_ID VARCHAR,
            MODEL_NAME VARCHAR,
            MODEL_VERSION VARCHAR,
            RUN_STATUS VARCHAR,
            MESSAGE VARCHAR,
            CREATED_AT TIMESTAMP_NTZ
                DEFAULT CURRENT_TIMESTAMP()
        )
        """
    ).collect()


def write_log(
    session: Session,
    run_id: str,
    status: str,
    message: str,
) -> None:
    safe_message = message.replace(
        "'",
        "''",
    )[:8000]

    session.sql(
        f"""
        INSERT INTO
        KMAT_COST_MODEL_DB.CORE_ML.
        BMCS_COMPOSITE_REGISTRATION_LOG_V2
        (
            RUN_ID,
            MODEL_NAME,
            MODEL_VERSION,
            RUN_STATUS,
            MESSAGE
        )
        VALUES
        (
            '{run_id}',
            '{MODEL_NAME}',
            '{NEW_VERSION_NAME}',
            '{status}',
            '{safe_message}'
        )
        """
    ).collect()


def validate_partition(
    frame: pd.DataFrame,
    partition_name: str,
) -> None:
    if len(frame) < 200:
        raise RuntimeError(
            f"{partition_name} contains only "
            f"{len(frame)} rows."
        )

    required_columns = (
        FEATURE_COLUMNS + [TARGET_COLUMN]
    )

    missing_columns = sorted(
        set(required_columns)
        .difference(frame.columns)
    )

    if missing_columns:
        raise RuntimeError(
            f"{partition_name} is missing columns: "
            f"{missing_columns}"
        )

    if frame[required_columns].isna().any().any():
        raise RuntimeError(
            f"{partition_name} contains null model values."
        )

    invalid_target = ~frame[
        TARGET_COLUMN
    ].isin([0, 1])

    if invalid_target.any():
        raise RuntimeError(
            f"{partition_name} contains invalid labels."
        )


def main(session: Session) -> str:
    session.use_database(DATABASE)
    session.use_schema(SCHEMA)

    run_id = (
        "BMCSCOMP_"
        + datetime.now(timezone.utc).strftime(
            "%Y%m%d_%H%M%S_%f"
        )
    )

    create_registration_log(session)

    write_log(
        session,
        run_id,
        "STARTED",
        "Composite BMCS model registration started.",
    )

    try:
        source = session.table(
            SOURCE_OBJECT
        ).to_pandas()

        source.columns = [
            str(column).upper()
            for column in source.columns
        ]

        source[TIMESTAMP_COLUMN] = pd.to_datetime(
            source[TIMESTAMP_COLUMN]
        )

        training = source[
            source[TIMESTAMP_COLUMN]
            < MODEL_TRAIN_END
        ].copy()

        calibration = source[
            (
                source[TIMESTAMP_COLUMN]
                >= CALIBRATION_START
            )
            & (
                source[TIMESTAMP_COLUMN]
                < CALIBRATION_END
            )
            & (
                source["DATA_SPLIT"]
                == "VALIDATION"
            )
        ].copy()

        validate_partition(
            training,
            "training",
        )

        validate_partition(
            calibration,
            "calibration",
        )

        base_pipeline = build_base_pipeline()

        base_pipeline.fit(
            training[FEATURE_COLUMNS],
            training[TARGET_COLUMN]
            .astype(int)
            .to_numpy(),
        )

        raw_calibration_probability = np.asarray(
            base_pipeline.predict_proba(
                calibration[FEATURE_COLUMNS]
            )[:, 1],
            dtype=float,
        )

        raw_calibration_probability = np.clip(
            raw_calibration_probability,
            PROBABILITY_EPSILON,
            1.0 - PROBABILITY_EPSILON,
        )

        isotonic_calibrator = IsotonicRegression(
            y_min=PROBABILITY_EPSILON,
            y_max=1.0 - PROBABILITY_EPSILON,
            out_of_bounds="clip",
        )

        isotonic_calibrator.fit(
            raw_calibration_probability,
            calibration[TARGET_COLUMN]
            .astype(int)
            .to_numpy(),
        )

        context = custom_model.ModelContext(
            base_pipeline=base_pipeline,
            isotonic_calibrator=isotonic_calibrator,
        )

        composite_model = (
            GovernedCalibratedBMCSModel(
                context=context,
            )
        )

        local_output = composite_model.predict(
            calibration[FEATURE_COLUMNS]
            .head(10)
        )

        expected_output_columns = [
            "RAW_CORRECT_MAPPING_PROBABILITY",
            "CALIBRATED_CORRECT_MAPPING_PROBABILITY",
            "ML_BMCS_SCORE",
        ]

        if list(local_output.columns) != (
            expected_output_columns
        ):
            raise RuntimeError(
                "Local composite output schema is invalid: "
                f"{list(local_output.columns)}"
            )

        if len(local_output) != 10:
            raise RuntimeError(
                "Local composite smoke test returned "
                "an invalid row count."
            )

        if not np.isfinite(
            local_output.to_numpy(dtype=float)
        ).all():
            raise RuntimeError(
                "Local composite output contains "
                "non-finite values."
            )

        metric_row = (
            session.table(
                "KMAT_COST_MODEL_DB.CORE_ML."
                "BMCS_FINAL_TEST_EVALUATION_V2"
            )
            .select(
                "ROC_AUC",
                "PR_AUC",
                "BRIER_SCORE",
                "LOG_LOSS",
                "ECE_10_BIN",
            )
            .collect()
        )

        if len(metric_row) != 1:
            raise RuntimeError(
                "BMCS_FINAL_TEST_EVALUATION_V2 must "
                "contain exactly one row."
            )

        metric_values = metric_row[0].as_dict()

        registered_metrics = {
            "ROC_AUC":
                float(metric_values["ROC_AUC"]),
            "PR_AUC":
                float(metric_values["PR_AUC"]),
            "BRIER_SCORE":
                float(metric_values["BRIER_SCORE"]),
            "LOG_LOSS":
                float(metric_values["LOG_LOSS"]),
            "ECE_10_BIN":
                float(metric_values["ECE_10_BIN"]),
        }

        registry = Registry(
            session=session,
            database_name=DATABASE,
            schema_name=SCHEMA,
        )

        registration_action = "REGISTERED"

        try:
            model_version = (
                registry
                .get_model(MODEL_NAME)
                .version(NEW_VERSION_NAME)
            )
            registration_action = (
                "ALREADY_REGISTERED"
            )

        except Exception:
            model_version = registry.log_model(
                composite_model,

                model_name=MODEL_NAME,
                version_name=NEW_VERSION_NAME,

                comment=(
                    "Atomic BMCS trust model containing "
                    "XGB_V1_BALANCED and isotonic calibration. "
                    "Synthetic shadow advisory only. "
                    "No automatic approval, rejection, "
                    "quotation or cost authority."
                ),

                metrics=registered_metrics,

                sample_input_data=(
                    calibration[FEATURE_COLUMNS]
                    .head(100)
                ),

                pip_requirements=[
                    f"pandas=={pd.__version__}",
                    f"numpy=={np.__version__}",
                    f"scikit-learn=={sklearn.__version__}",
                    f"xgboost=={xgboost.__version__}",
                ],

                artifact_repository_map={
                    "pip":
                        "snowflake.snowpark."
                        "pypi_shared_repository"
                },

                target_platforms=["WAREHOUSE"],
            )

        warehouse_output = model_version.run(
            calibration[FEATURE_COLUMNS]
            .head(5),
            function_name="predict",
        )

        if hasattr(
            warehouse_output,
            "to_pandas",
        ):
            warehouse_output_pdf = (
                warehouse_output.to_pandas()
            )
        else:
            warehouse_output_pdf = pd.DataFrame(
                warehouse_output
            )

        warehouse_output_pdf.columns = [
            str(column).upper()
            for column in warehouse_output_pdf.columns
        ]

        if len(warehouse_output_pdf) != 5:
            raise RuntimeError(
                "Warehouse smoke test did not return "
                "five rows."
            )

        missing_output_columns = sorted(
            set(expected_output_columns)
            .difference(
                warehouse_output_pdf.columns
            )
        )

        if missing_output_columns:
            raise RuntimeError(
                "Warehouse inference is missing outputs: "
                f"{missing_output_columns}; returned="
                f"{list(warehouse_output_pdf.columns)}"
            )

        session.sql(
            f"""
            ALTER MODEL
            {DATABASE}.{SCHEMA}.{MODEL_NAME}
            SET DEFAULT_VERSION = '{NEW_VERSION_NAME}'
            """
        ).collect()

        success_message = (
            f"{registration_action}; "
            f"training_rows={len(training)}; "
            f"calibration_rows={len(calibration)}; "
            f"warehouse_smoke_rows="
            f"{len(warehouse_output_pdf)}; "
            f"default_version={NEW_VERSION_NAME}; "
            f"sklearn={sklearn.__version__}; "
            f"xgboost={xgboost.__version__}"
        )

        write_log(
            session,
            run_id,
            "SUCCESS",
            success_message,
        )

        return json.dumps(
            {
                "status": "SUCCESS",

                "registration_action":
                    registration_action,

                "model_name":
                    MODEL_NAME,

                "model_version":
                    NEW_VERSION_NAME,

                "default_version":
                    NEW_VERSION_NAME,

                "training_rows":
                    int(len(training)),

                "calibration_rows":
                    int(len(calibration)),

                "warehouse_smoke_rows":
                    int(len(warehouse_output_pdf)),

                "output_columns":
                    expected_output_columns,

                "deployment_mode":
                    "SHADOW",

                "ml_advisory_only":
                    True,

                "automatic_approval_enabled":
                    False,

                "automatic_rejection_enabled":
                    False,

                "official_cost_impact_allowed":
                    False,

                "business_decision_allowed":
                    False,
            },
            sort_keys=True,
        )

    except Exception as exc:
        write_log(
            session,
            run_id,
            "FAILED",
            f"{type(exc).__name__}: {str(exc)}",
        )
        raise


In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()
session.sql("USE DATABASE KMAT_COST_MODEL_DB").collect()
session.sql("USE SCHEMA CORE_ML").collect()

# Fix string annotations caused by 'from __future__ import annotations'
# in a previously-run cell polluting the kernel's __main__ module
GovernedCalibratedBMCSModel.predict.__annotations__['input_df'] = pd.DataFrame
GovernedCalibratedBMCSModel.predict.__annotations__['return'] = pd.DataFrame

result = main(session)
print(result)

In [ ]:
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from snowflake.snowpark import Session
from snowflake.ml.registry import Registry

DATABASE = "KMAT_COST_MODEL_DB"
SCHEMA = "CORE_ML"
SCORING_BATCH_ID = "BMCS_PHASE7_SHADOW_001"

INPUT_OBJECT = (
    "KMAT_COST_MODEL_DB.CORE_ML."
    "BMCS_PHASE7_SCORING_INPUT_FINAL_V2"
)
OUTPUT_OBJECT = (
    "KMAT_COST_MODEL_DB.CORE_ML."
    "BMCS_PHASE7_SHADOW_SCORE_FINAL_V2"
)

MODEL_NAME = "BMCS_CALIBRATED_CLASSIFIER"
MODEL_VERSION = "V2_SHADOW_2"
FEATURE_SET_VERSION = "BMCS_PHASE7_ADAPTER_FINAL_V2"

LOW_ADVISORY_THRESHOLD = 0.485
HIGH_ADVISORY_THRESHOLD = 0.945

CATEGORICAL_COLUMNS = [
    "SOURCE_LANGUAGE",
    "EXTRACTED_ENGINE",
    "EXTRACTED_CAB",
    "EXTRACTED_WHEEL",
    "EXTRACTED_COLOR",
    "CORTEX_EXTRACTION_STATUS",
]

NUMERIC_COLUMNS = [
    "DOCUMENT_LENGTH_CHARS",
    "PAGE_COUNT",
    "OCR_QUALITY_SCORE",
    "DOCUMENT_COMPLEXITY_SCORE",
    "AMBIGUITY_TERM_COUNT",
    "CONFLICTING_PHRASE_COUNT",
    "UNSUPPORTED_VALUE_COUNT",
    "SYNONYM_MATCH_STRENGTH",
    "TRANSLATION_USED_FLAG",
    "TRANSLATION_QUALITY_SCORE",
    "SOURCE_TEXT_PRESENT_FLAG",
    "ENGINE_EXTRACTION_CONFIDENCE",
    "CAB_EXTRACTION_CONFIDENCE",
    "WHEEL_EXTRACTION_CONFIDENCE",
    "COLOR_EXTRACTION_CONFIDENCE",
    "ENGINE_TEXT_EVIDENCE_FLAG",
    "CAB_TEXT_EVIDENCE_FLAG",
    "WHEEL_TEXT_EVIDENCE_FLAG",
    "COLOR_TEXT_EVIDENCE_FLAG",
    "VALID_EXTRACTED_VALUE_COUNT",
    "TEXT_SUPPORTED_VALUE_COUNT",
    "MISSING_REQUIREMENT_COUNT",
    "EXACT_PHRASE_MATCH_COUNT",
    "SYNONYM_MATCH_COUNT",
    "RULE_VALIDATION_PASS_COUNT",
    "CORTEX_REASONING_AVAILABLE_FLAG",
    "RULE_BMCS_SCORE",
    "CORTEX_CONFIDENCE_SCORE",
    "DEVELOPMENT_HYBRID_BMCS_SCORE",
]

FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS


def create_run_log(session):
    session.sql(
        """
        CREATE TABLE IF NOT EXISTS
        KMAT_COST_MODEL_DB.CORE_ML.BMCS_PHASE7_SHADOW_RUN_LOG_FINAL_V2
        (
            RUN_ID VARCHAR,
            SCORING_BATCH_ID VARCHAR,
            RUN_STATUS VARCHAR,
            MESSAGE VARCHAR,
            CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
        )
        """
    ).collect()


def log_status(session, run_id, status, message):
    safe_message = message.replace("'", "''")[:8000]
    safe_batch = SCORING_BATCH_ID.replace("'", "''")
    session.sql(
        f"""
        INSERT INTO
        KMAT_COST_MODEL_DB.CORE_ML.BMCS_PHASE7_SHADOW_RUN_LOG_FINAL_V2
        (RUN_ID, SCORING_BATCH_ID, RUN_STATUS, MESSAGE)
        VALUES ('{run_id}', '{safe_batch}', '{status}', '{safe_message}')
        """
    ).collect()


def normalize_model_output(result, expected_rows):
    if hasattr(result, "to_pandas"):
        output = result.to_pandas()
    else:
        output = pd.DataFrame(result)
    output.columns = [str(c).upper() for c in output.columns]
    required = [
        "RAW_CORRECT_MAPPING_PROBABILITY",
        "CALIBRATED_CORRECT_MAPPING_PROBABILITY",
        "ML_BMCS_SCORE",
    ]
    missing = [c for c in required if c not in output.columns]
    if missing:
        raise RuntimeError(f"Model output missing {missing}; got={list(output.columns)}")
    if len(output) != expected_rows:
        raise RuntimeError(f"Row count mismatch: expected {expected_rows}, got {len(output)}")
    output = output[required].apply(pd.to_numeric, errors="coerce")
    if output.isna().any().any():
        raise RuntimeError("Model returned null or non-numeric outputs.")
    if not np.isfinite(output.to_numpy(dtype=float)).all():
        raise RuntimeError("Model returned non-finite outputs.")
    if not output[required[0]].between(0.0, 1.0).all():
        raise RuntimeError("Raw probabilities outside 0-1.")
    if not output[required[1]].between(0.0, 1.0).all():
        raise RuntimeError("Calibrated probabilities outside 0-1.")
    if not output[required[2]].between(0.0, 100.0).all():
        raise RuntimeError("ML BMCS scores outside 0-100.")
    return output.reset_index(drop=True)


def build_output_row(source_row, raw_probability, calibrated_probability, ml_score, trust_band, fallback_reason):
    return {
        "SCORING_BATCH_ID": SCORING_BATCH_ID,
        "ENTITY_ID": source_row["ENTITY_ID"],
        "RFQ_ID": source_row["RFQ_ID"],
        "SIMULATION_ID": source_row["SIMULATION_ID"],
        "KMAT_ID": source_row["KMAT_ID"],
        "CONFIGURATION_VERSION": source_row["CONFIGURATION_VERSION"],
        "RAW_ML_PROBABILITY": raw_probability,
        "CALIBRATED_ML_PROBABILITY": calibrated_probability,
        "ML_BMCS_SCORE": ml_score,
        "ML_ADVISORY_TRUST_BAND": trust_band,
        "ML_FALLBACK_REASON": fallback_reason,
        "MODEL_OOD_FLAG": bool(source_row["MODEL_OOD_FLAG"]),
        "PROXY_FEATURE_COUNT": int(source_row["PROXY_FEATURE_COUNT"]),
        "SHADOW_REVIEW_ACTION": "ENGINEERING_REVIEW_REQUIRED",
        "OFFICIAL_TRUST_SOURCE": "PHASE7_RULE_CORTEX_HYBRID",
        "OFFICIAL_RULE_BMCS_SCORE": source_row["OFFICIAL_RULE_BMCS_SCORE"],
        "OFFICIAL_CORTEX_BMCS_SCORE": source_row["OFFICIAL_CORTEX_BMCS_SCORE"],
        "OFFICIAL_HYBRID_BMCS_SCORE": source_row["OFFICIAL_HYBRID_BMCS_SCORE"],
        "OFFICIAL_REVIEW_STATUS": source_row["OFFICIAL_REVIEW_STATUS"],
        "OFFICIAL_REVIEW_REQUIRED_FLAG": source_row["OFFICIAL_REVIEW_REQUIRED_FLAG"],
        "OFFICIAL_TRUSTED_COST_ALLOWED_FLAG": source_row["OFFICIAL_TRUSTED_COST_ALLOWED_FLAG"],
        "OFFICIAL_BMCS_METHOD": source_row["OFFICIAL_BMCS_METHOD"],
        "PHYSICAL_MODEL_NAME": MODEL_NAME,
        "PHYSICAL_MODEL_VERSION": MODEL_VERSION,
        "FEATURE_SET_VERSION": FEATURE_SET_VERSION,
        "DEPLOYMENT_MODE": "SHADOW",
        "ML_ADVISORY_ONLY_FLAG": True,
        "OFFICIAL_COST_IMPACT_ALLOWED_FLAG": False,
        "BUSINESS_DECISION_ALLOWED_FLAG": False,
    }


def main(session):
    session.use_database(DATABASE)
    session.use_schema(SCHEMA)

    run_id = "BMCSP7_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
    create_run_log(session)
    log_status(session, run_id, "STARTED", "Phase 7 BMCS shadow scoring started.")

    safe_batch = SCORING_BATCH_ID.replace("'", "''")
    temp_table = "BMCS_PHASE7_SCORE_TEMP_" + datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S%f")

    try:
        input_pdf = session.sql(
            f"SELECT * FROM {INPUT_OBJECT} WHERE SCORING_BATCH_ID = '{safe_batch}' AND INPUT_STATUS = 'PENDING' ORDER BY ENTITY_ID"
        ).to_pandas()
        input_pdf.columns = [str(c).upper() for c in input_pdf.columns]

        if input_pdf.empty:
            log_status(session, run_id, "SUCCESS", "No pending rows were available.")
            return json.dumps({"status": "NO_PENDING_ROWS", "scoring_batch_id": SCORING_BATCH_ID}, sort_keys=True)

        eligible = input_pdf[input_pdf["FEATURE_QUALITY_PASS_FLAG"] == True].copy().reset_index(drop=True)
        fallback = input_pdf[input_pdf["FEATURE_QUALITY_PASS_FLAG"] != True].copy().reset_index(drop=True)

        output_rows = []

        if not eligible.empty:
            registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)
            model_version = registry.get_model(MODEL_NAME).version(MODEL_VERSION)
            model_result = model_version.run(eligible[FEATURE_COLUMNS], function_name="predict", strict_input_validation=True)
            predictions = normalize_model_output(model_result, len(eligible))

            for index, row in eligible.iterrows():
                raw_prob = float(predictions.iloc[index]["RAW_CORRECT_MAPPING_PROBABILITY"])
                cal_prob = float(predictions.iloc[index]["CALIBRATED_CORRECT_MAPPING_PROBABILITY"])
                ml_score = float(predictions.iloc[index]["ML_BMCS_SCORE"])

                if bool(row["MODEL_OOD_FLAG"]):
                    trust_band = "OOD_ADVISORY"
                elif cal_prob >= HIGH_ADVISORY_THRESHOLD:
                    trust_band = "HIGH_CONFIDENCE_ADVISORY"
                elif cal_prob <= LOW_ADVISORY_THRESHOLD:
                    trust_band = "LOW_CONFIDENCE_ADVISORY"
                else:
                    trust_band = "MEDIUM_CONFIDENCE_ADVISORY"

                output_rows.append(build_output_row(row, raw_prob, cal_prob, ml_score, trust_band, None))

        for _, row in fallback.iterrows():
            output_rows.append(build_output_row(row, None, None, None, "DATA_QUALITY_FALLBACK", row["ADAPTER_STATUS"]))

        result_pdf = pd.DataFrame(output_rows)

        session.sql(f"DELETE FROM {OUTPUT_OBJECT} WHERE SCORING_BATCH_ID='{safe_batch}'").collect()

        session.write_pandas(
            result_pdf,
            table_name=temp_table,
            database=DATABASE,
            schema=SCHEMA,
            auto_create_table=True,
            overwrite=True,
            quote_identifiers=False,
            table_type="transient",
        )

        # Use CURRENT_TIMESTAMP() for SCORED_AT to avoid pandas->Snowflake type mismatch
        session.sql(
            f"""
            INSERT INTO {OUTPUT_OBJECT}
            SELECT
                SCORING_BATCH_ID, ENTITY_ID, RFQ_ID, SIMULATION_ID, KMAT_ID,
                CONFIGURATION_VERSION, RAW_ML_PROBABILITY, CALIBRATED_ML_PROBABILITY,
                ML_BMCS_SCORE, ML_ADVISORY_TRUST_BAND, ML_FALLBACK_REASON,
                MODEL_OOD_FLAG, PROXY_FEATURE_COUNT, SHADOW_REVIEW_ACTION,
                OFFICIAL_TRUST_SOURCE, OFFICIAL_RULE_BMCS_SCORE,
                OFFICIAL_CORTEX_BMCS_SCORE, OFFICIAL_HYBRID_BMCS_SCORE,
                OFFICIAL_REVIEW_STATUS, OFFICIAL_REVIEW_REQUIRED_FLAG,
                OFFICIAL_TRUSTED_COST_ALLOWED_FLAG, OFFICIAL_BMCS_METHOD,
                PHYSICAL_MODEL_NAME, PHYSICAL_MODEL_VERSION, FEATURE_SET_VERSION,
                DEPLOYMENT_MODE, ML_ADVISORY_ONLY_FLAG,
                OFFICIAL_COST_IMPACT_ALLOWED_FLAG, BUSINESS_DECISION_ALLOWED_FLAG,
                CURRENT_TIMESTAMP()
            FROM {DATABASE}.{SCHEMA}.{temp_table}
            """
        ).collect()

        session.sql(
            f"""
            UPDATE {INPUT_OBJECT}
            SET INPUT_STATUS = IFF(FEATURE_QUALITY_PASS_FLAG = TRUE, 'SCORED', 'FALLBACK'),
                UPDATED_AT = CURRENT_TIMESTAMP()
            WHERE SCORING_BATCH_ID = '{safe_batch}' AND INPUT_STATUS = 'PENDING'
            """
        ).collect()

        session.sql(f"DROP TABLE IF EXISTS {DATABASE}.{SCHEMA}.{temp_table}").collect()

        message = f"input_rows={len(input_pdf)}; ml_scored={len(eligible)}; fallback={len(fallback)}; output={len(result_pdf)}; model={MODEL_NAME}/{MODEL_VERSION}"
        log_status(session, run_id, "SUCCESS", message)

        return json.dumps({
            "status": "SUCCESS",
            "scoring_batch_id": SCORING_BATCH_ID,
            "input_rows": int(len(input_pdf)),
            "ml_scored_rows": int(len(eligible)),
            "fallback_rows": int(len(fallback)),
            "output_rows": int(len(result_pdf)),
            "physical_model_name": MODEL_NAME,
            "physical_model_version": MODEL_VERSION,
            "deployment_mode": "SHADOW",
            "ml_advisory_only": True,
            "business_decision_allowed": False,
        }, sort_keys=True)

    except Exception as exc:
        try:
            session.sql(f"DROP TABLE IF EXISTS {DATABASE}.{SCHEMA}.{temp_table}").collect()
        except Exception:
            pass
        log_status(session, run_id, "FAILED", f"{type(exc).__name__}: {str(exc)}")
        raise


from snowflake.snowpark.context import get_active_session

session = get_active_session()
result = main(session)
print(result)